# 01 · Perfilado de los datos (fase *Prepare*)

> **PRUEBA PRELIMINAR (bitácora D-09, D-11 y D-13).** Se usa la ventana **2024-11 a 2026-10, 24 meses (D-06)**. Las cifras son resultados de la prueba preliminar y deben revisarse antes de copiarse a documentos finales.

**¿Qué hace este notebook?** Obtiene los datos de viajes de ECOBICI, los guarda en un formato más rápido (capa *bronze*) **sin cambiar nada**, comprueba que no se perdió ninguna fila y mide la calidad de cada mes.

**¿Por qué?** Antes de limpiar hay que ver los datos tal como vienen. Si se mezcla el diagnóstico con la limpieza, después no se puede justificar cada cambio (bitácora D-01).

In [ ]:
import sys
from pathlib import Path

# Localiza la raíz del repositorio (la carpeta que contiene config/pipeline.toml) para poder importar el pipeline.
raiz = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "config" / "pipeline.toml").exists())
sys.path.insert(0, str(raiz))

import duckdb
import matplotlib.pyplot as plt
import pandas as pd

from pipeline import bronze, config, descarga, gold, silver

cfg = config.cargar()
pd.options.display.float_format = "{:,.1f}".format
pd.options.display.max_columns = 30
pd.options.display.width = 200
print("Ventana de datos:", cfg["ventana"]["desde"], "a", cfg["ventana"]["hasta"])

## 1. Descargar los archivos del portal

El portal de datos abiertos publica un archivo CSV por mes, pero con nombres distintos cada vez. Por eso el programa **lee la página del portal**, encuentra el enlace de cada mes de la ventana y descarga solo lo que falta. Al terminar verifica que el tamaño del archivo coincide con el que anuncia el servidor, para detectar descargas incompletas. Si el archivo ya está, se omite.

In [ ]:
estado = descarga.descargar_ventana(cfg)
pd.DataFrame(estado, columns=["mes", "estado", "bytes"]).assign(MB=lambda d: d["bytes"] / 1e6)[["mes", "estado", "MB"]]

## 2. Pasar los CSV a la capa *bronze* y validar el conteo

Cada CSV se convierte a Parquet (un formato comprimido y rápido) **leyendo todo como texto**. Así no se pierden los ceros de los códigos de estación (`015`) ni se esconden valores raros. Para estar seguros de que no se perdió nada, las filas se cuentan **dos veces por separado**: una leyendo el CSV directamente y otra en el Parquet resultante. Si no coinciden en algún mes, el proceso se detiene.

_Esta comprobación existe porque en la exploración previa DuckDB omitió dos meses completos sin avisar (bitácora D-03)._

In [ ]:
conteos = pd.DataFrame(bronze.construir_bronze(cfg))
print("Filas CSV = filas bronze en todos los meses:", bool(conteos["coincide"].all()))
conteos.style.format({"filas_csv": "{:,}", "filas_bronze": "{:,}"})

## 3. Perfil de calidad por mes

Para cada mes se mide: cuántos viajes hay, cómo se reparte el género, cuántas edades vienen vacías o con decimal, cuántas estaciones distintas aparecen, y cuántos viajes usan códigos especiales (**códigos dobles** como `271-272`, que son estaciones contiguas, y estaciones `Temporal`). Aquí solo se **mide**; nada se corrige todavía.

In [ ]:
con = duckdb.connect()
b = [(config.BRONZE / f"{m}.parquet").as_posix() for m in config.meses(cfg)]
perfil = con.execute(f"""
    SELECT mes_archivo AS mes, count(*) AS viajes,
           round(100 * avg((coalesce(genero, '') = 'M')::INT), 1) AS pct_hombres,
           round(100 * avg((coalesce(genero, '') = 'F')::INT), 1) AS pct_mujeres,
           round(100 * avg((coalesce(genero, '') = 'O')::INT), 2) AS pct_otro,
           round(100 * avg((coalesce(genero, '') = '?')::INT), 2) AS pct_desconocido,
           sum((edad IS NULL)::INT) AS edad_vacia,
           sum((edad LIKE '%.%')::INT) AS edad_con_decimal,
           count(DISTINCT est_retiro) AS estaciones_retiro,
           sum((est_retiro LIKE '%-%' OR est_arribo LIKE '%-%')::INT) AS viajes_con_codigo_doble,
           sum((est_retiro LIKE 'Temporal%' OR est_arribo LIKE 'Temporal%')::INT) AS viajes_estacion_temporal
    FROM read_parquet({b}) GROUP BY 1 ORDER BY 1""").df()
config.PERFIL.mkdir(parents=True, exist_ok=True)
perfil.to_csv(config.PERFIL / "calidad_por_mes.csv", index=False)
perfil.style.format({"viajes": "{:,}", "edad_vacia": "{:,}", "edad_con_decimal": "{:,}",
                     "viajes_con_codigo_doble": "{:,}", "viajes_estacion_temporal": "{:,}"})

In [ ]:
config.FIGURAS.mkdir(parents=True, exist_ok=True)
fig, ax = plt.subplots(figsize=(11, 4))
ax.bar(perfil["mes"], perfil["viajes"] / 1e6, color="#2a6fb0")
ax.set_ylabel("Millones de viajes")
ax.set_title("Viajes por mes de archivo")
ax.tick_params(axis="x", rotation=90)
ax.set_ylim(0, perfil["viajes"].max() / 1e6 * 1.15)
for x, v in zip(perfil["mes"], perfil["viajes"] / 1e6):
    ax.text(x, v + 0.02, f"{v:.2f}", ha="center", fontsize=7)
fig.tight_layout()
fig.savefig(config.FIGURAS / "01_viajes_por_mes.png", dpi=150)
plt.show()

## 4. Qué encontramos

- El esquema es el mismo en todos los meses de la ventana (se valida el encabezado de cada archivo).
- Desde 2026-06 la **edad llega con decimal** (`26.0`); en la fase de limpieza se convierte con una sola regla (bitácora D-07).
- Hay **códigos dobles** (`271-272`) y estaciones `Temporal`; se revisan en la limpieza y en la unión con el catálogo.
- Los archivos se organizan por fecha de **arribo**: cada mes trae algunos retiros del mes anterior.

**Siguiente paso:** `02_limpieza.ipynb` aplica las reglas de limpieza y registra cuántos viajes se excluyen y por qué.